# DA6401 — Kaggle Account 3: Localization (Task 2)

**Before running:**
- Accelerator → **P100**
- Internet → **ON**
- Then: Run All → close browser → come back later

W&B tag: `kaggle-sys3`

Estimated time: ~2 hours on P100

> Note: Trains localizer from scratch (no pretrained encoder). Once classifier.pth
> is available from Account 1 or 2, you can optionally re-run with it.

In [ ]:
import os
os.environ["WANDB_API_KEY"]      = "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"
os.environ["WANDB_DISABLE_CODE"] = "1"

CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

!git clone https://github.com/usnaveen/A2_Deep_Learning.git da6401_assignment_2 2>&1 | tail -2
%cd da6401_assignment_2
!pip install -q wandb albumentations scikit-learn

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable GPU!'}")

In [ ]:
from data.pets_dataset import download_oxford_pet
download_oxford_pet("./data/oxford_pet")
print("Dataset ready.")

In [ ]:
# ── Optional: if you have classifier.pth from Account 1 or 2, upload it here ──
# Upload the file manually via Kaggle's "Add data" or leave blank to train from scratch
CLASSIFIER_PATH = ""  # e.g. "/kaggle/input/your-dataset/classifier.pth"

if CLASSIFIER_PATH:
    import shutil
    shutil.copy(CLASSIFIER_PATH, "/kaggle/working/checkpoints/classifier.pth")
    print(f"Copied classifier.pth → will use pretrained encoder")
else:
    print("No classifier.pth provided — training localizer from scratch")

In [ ]:
# ── Localization training  (W&B tag: kaggle-sys3) ──
!python train.py \
    --task localize \
    --experiment kaggle-sys3 \
    --epochs 120 \
    --lr 1e-3 \
    --batch-size 32 \
    --num-workers 2 \
    --checkpoint-dir /kaggle/working/checkpoints

In [ ]:
# ── Final check: best IoU on validation set ──
import torch
from data.pets_dataset import create_dataloaders
from models.localization import VGG11Localizer

def compute_iou(pred, gt, eps=1e-6):
    px1, py1 = pred[:,0]-pred[:,2]/2, pred[:,1]-pred[:,3]/2
    px2, py2 = pred[:,0]+pred[:,2]/2, pred[:,1]+pred[:,3]/2
    tx1, ty1 = gt[:,0]-gt[:,2]/2,   gt[:,1]-gt[:,3]/2
    tx2, ty2 = gt[:,0]+gt[:,2]/2,   gt[:,1]+gt[:,3]/2
    ix1, iy1 = torch.max(px1,tx1), torch.max(py1,ty1)
    ix2, iy2 = torch.min(px2,tx2), torch.min(py2,ty2)
    inter = (ix2-ix1).clamp(0)*(iy2-iy1).clamp(0)
    union = (px2-px1).clamp(0)*(py2-py1).clamp(0)+(tx2-tx1).clamp(0)*(ty2-ty1).clamp(0)-inter
    return inter/(union+eps)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, val_loader, _ = create_dataloaders(root="./data/oxford_pet", batch_size=64, num_workers=2)

model = VGG11Localizer().to(device)
model.load_state_dict(torch.load("/kaggle/working/checkpoints/localizer.pth", map_location=device, weights_only=False))
model.eval()

total_iou, total = 0.0, 0
with torch.no_grad():
    for batch in val_loader:
        pred = model(batch["image"].to(device)).cpu()
        ious = compute_iou(pred, batch["bbox"])
        total_iou += ious.sum().item()
        total += len(ious)

print(f"\n✅ Val mean IoU = {total_iou/total:.4f}")
print("   Checkpoint saved at: /kaggle/working/checkpoints/localizer.pth")
print("   Download this file from Kaggle output tab!")